In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import tsplib95

from clonalg.antibody.permutation_antibody import PermutationAntibody, PermutationAntibodyBuilder
from clonalg.model.clonalg_optimization import OptimizationClonalg
from clonalg.problems.problem import Rastrigin
from clonalg.problems.visualization import plot_3d_surface, plot_contour_and_paths

In [4]:
def gap(found: float, optimal: int) -> float:
    return (found - optimal) / optimal * 100

In [5]:
def load_solutions(filepath: str) -> dict[str, int]:
    solutions = {}
    for line in Path(filepath).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        name, _, rest = line.partition(":")
        solutions[name.strip().lower()] = int(rest.strip())
    return solutions
#

solutions = load_solutions('./data/solutions')

In [6]:
def tour_cost(tour: np.ndarray, problem: tsplib95.models.StandardProblem) -> int:
    n = len(tour)
    return sum(
        problem.get_weight(int(tour[i]), int(tour[(i + 1) % n]))
        for i in range(n)
    )

In [7]:
PROBLEM_NAME='bays29'

problem = tsplib95.load(f'./data/{PROBLEM_NAME}.tsp')
problem.dimension # cities are labeled from 1 to N

29

In [8]:
n_cities = problem.dimension
bounds = [(2, problem.dimension)] * (n_cities - 1)

def factory() -> PermutationAntibody:
    genes = np.random.permutation(np.arange(2, n_cities + 1))
    return (
        PermutationAntibodyBuilder()
        .with_genes(genes)
        .with_cost_fn(lambda g: tour_cost(np.concatenate(([1], g)), problem))
        .build()
    )

clonalg = OptimizationClonalg(
    population_size=20,
    clone_factor=0.2,
    n_replace=5,
    n_generations=1000,
    antibody_factory=factory,
    suppression_threshold=1
)

memory = clonalg.run()
memory.sort(key=lambda ab: ab.affinity(None), reverse=True)


  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:42<00:00, 23.42it/s]


In [9]:
best = memory[0]
best_tour = np.concatenate(([1], best.genes))

# print(f"Best solution: {best_tour}")

print(f"Tour cost = {tour_cost(best_tour, problem)}")
print(f"Best tour cost = {solutions[PROBLEM_NAME]}")

print(f"Affinity = {best.affinity(None):.6f}")
print(f"Gap from optimal: {gap(tour_cost(best_tour, problem), solutions[PROBLEM_NAME]):.2f}%")

Tour cost = 3517
Best tour cost = 2020
Affinity = 0.000284
Gap from optimal: 74.11%


In [10]:
tour_cost(best_tour, problem)


3517